In [1]:
import sys 
sys.path.append('/home/hydrogen/workspace/Space_GW/pespace')
sys.path.append('/home/hydrogen/workspace/Space_GW/wf4ti')

import taichi as ti
ti.init()
import numpy as np
import h5py
from matplotlib import pyplot as plt
%matplotlib inline


[Taichi] version 1.6.0, llvm 15.0.4, commit f1c6fbbd, linux, python 3.10.12


[I 07/31/24 16:08:28.928 2046237] [shell.py:_shell_pop_print@23] Graphical python shell detected, using wrapped sys.stdout


[Taichi] Starting on arch=x64


In [2]:
# file_path = "/home/hydrogen/workspace/Space_GW/LDC/LDC_data/LDC1-1_MBHB_v2_TD_9gc2s16.hdf5"
file_path = "/home/hydrogen/workspace/Space_GW/LDC/LDC_data/LDC1-1_MBHB_v1_1_FD.hdf5"
with h5py.File(file_path, "r") as f:
    # f.visit(lambda name: print(name))

    source_parameters = {}
    print('Source information: ')
    for parameter, value in f["H5LISA/GWSources/MBHB-0"].items():
        print(f"{parameter}: {value[()]}")
        source_parameters[parameter] = value[()]

    # print(f["H5LISA/PreProcess/TDIGenerator"][()])
    TDI_data = f["H5LISA/PreProcess/TDIdata"][()]
    print(TDI_data)
    print(TDI_data.shape)


Source information: 
Approximant: b'IMRPhenomD'
AzimuthalAngleOfSpin1: 0.6171792478977071
AzimuthalAngleOfSpin2: 4.75979656623224
Cadence: 10.0
CoalescenceTime: 25135000.0
Distance: 60.42017466175677
EclipticLatitude: -0.5256036732051035
EclipticLongitude: 1.1637
InitialAzimuthalAngleL: 0.30782038099413395
InitialPolarAngleL: 1.2498
Mass1: 2803843.776
Mass2: 285210.246
ObservationDuration: 41943040.0
PhaseAtCoalescence: 2.596553404898615
PolarAngleOfSpin1: 0.0
PolarAngleOfSpin2: 0.0
Redshift: 6.1178
Spin1: 0.8986046314480332
Spin2: 0.9465882372512956
hphcData: [[ 0.00000000e+00 -2.52087943e-22 -8.74551457e-21]
 [ 1.00000000e+01 -2.30277543e-22 -8.74164257e-21]
 [ 2.00000000e+01 -2.08463967e-22 -8.73765719e-21]
 ...
 [ 4.19430100e+07  0.00000000e+00  0.00000000e+00]
 [ 4.19430200e+07  0.00000000e+00  0.00000000e+00]
 [ 4.19430300e+07  0.00000000e+00  0.00000000e+00]]
[[ 1.00000000e+01  1.43833862e-20  2.60735825e-20 -3.76029472e-20]
 [ 2.00000000e+01 -4.00786179e-20 -1.10577882e-20  8.3

In [5]:
time_samples = TDI_data[:,0]
TDI_strain = TDI_data[:,1:]
print(TDI_strain)
print(TDI_strain.shape)
TDI_strain = TDI_strain.T
print(TDI_strain)
print(TDI_strain.shape)

duration = time_samples[-1]-time_samples[0]
cadence = time_samples[1]-time_samples[0]
print('duration: ', duration)
print('cadence: ', cadence)



[[ 1.43833862e-20  2.60735825e-20 -3.76029472e-20]
 [-4.00786179e-20 -1.10577882e-20  8.30232345e-20]
 [ 5.03369370e-20 -4.04076758e-21 -8.91464428e-20]
 ...
 [-9.93635288e-21  5.14937370e-21 -3.06227025e-21]
 [-4.46615077e-21  9.52534556e-21  3.00762049e-20]
 [-1.15327979e-21 -2.70341673e-20 -3.94044645e-22]]
(4194304, 3)
[[ 1.43833862e-20 -4.00786179e-20  5.03369370e-20 ... -9.93635288e-21
  -4.46615077e-21 -1.15327979e-21]
 [ 2.60735825e-20 -1.10577882e-20 -4.04076758e-21 ...  5.14937370e-21
   9.52534556e-21 -2.70341673e-20]
 [-3.76029472e-20  8.30232345e-20 -8.91464428e-20 ... -3.06227025e-21
   3.00762049e-20 -3.94044645e-22]]
(3, 4194304)
duration:  41943030.0
cadence:  10.0


In [ ]:
from pespace.detectors import TDIChannelsData
from pespace.noise import noise_models

mbhb = TDIChannelsData()
mbhb.set_frequency_domain_data_from_input_array(channels=( "X", "Y", "Z"), generation='1.5', duration=duration, cadence=cadence, )
mbhb.set_frequency_domain_data_with_zero_value(channels=("A", "E", "T", "X", "Y", "Z"), generation='1.5', duration=2592000, cadence=10)
mbhb.set_frequency_domain_noise_power_density_from_analystic_model(noise_models['LISA_SciRDv1'])
noise_realization = mbhb.generate_realization_from_frequency_domain_noise_power_density()
mbhb.add_into_frequency_domian_data(noise_realization)
